# Module 1 — Business Understanding & Data Generation
**Project:** Learner Behavior Segmentation for an Online Learning Platform

**Business question.** *What distinct behavioural profiles exist among our learners, and how should
content, pacing or support be tailored to each group?*

This notebook (Data Engineer role) covers:
1. Business problem, decision makers and KPIs (engagement score, at-risk learner rate, intervention cost)
2. Acquisition and integrity check of the OULAD base data
3. Big-data ETL of the 10.6M-row clickstream with **PySpark** (DataFrame API + Pair RDD, Lectures 5–6)
4. Integration of the seven OULAD tables at enrollment grain
5. Generation of the **synthetic behavioural extension** (Faker + conditional business logic)
6. Validation of the synthetic data's realism and the **data dictionary**

| Analytics type (Lecture 1) | Where it appears in this project |
|---|---|
| Descriptive | EDA, OLAP cube, cluster profiles (Modules 2, 5) |
| Predictive | Segment assignment of new learners (API), surrogate decision tree, drift monitoring (Modules 5–7) |
| Prescriptive | Decision-tree expected value and LP budget allocation of interventions (Module 8) |

The workflow follows the KDD process (Lecture 8): *selection → preprocessing → transformation → data mining → interpretation/evaluation*.

In [1]:
import sys, json, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src")) if (Path.cwd().parent / "src").exists() else sys.path.insert(0, "src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from edtech import config as C, etl_spark, oulad, synthetic, data_dictionary, utils as U, viz as V
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
print("Project root:", C.ROOT)

Project root: E:\20261\BA\midterm-project


## 1. Business understanding

**Decision makers:** Head of Learner Success (support budget), course designers (content & pacing),
tutors (who to contact and when).

**Decisions supported**
* Which learners receive which support (nudge, coaching, tutor call, peer group)?
* How should content and deadlines be paced for each behavioural group?
* When is the segmentation outdated and must be rebuilt?

**KPIs** (definitions are implemented in `src/edtech/features.py` and documented in the data dictionary)

| KPI | Definition | Target direction |
|---|---|---|
| Engagement score (0–100) | 100 × (0.30·pct(active-day fraction) + 0.20·pct(logins/week) + 0.20·video completion + 0.20·submission rate + 0.10·pct(forum posts/week)) | ↑ |
| At-risk learner rate | share of enrollments ending *Fail* or *Withdrawn* (outcome, never a model input) | ↓ |
| Intervention cost | Σ learners × unit cost of the intervention they receive (catalogue below) | minimise per extra pass |
| Cost per additional pass | intervention cost ÷ expected additional passes | ↓ |
| Segment coverage | share of learners with a confident segment assignment | ↑ |

In [2]:
catalogue = pd.DataFrame([(k, *v) for k, v in C.INTERVENTION_CATALOGUE.items()],
                         columns=["intervention", "cost_usd_per_learner", "tutor_minutes", "assumed_base_uplift_pass_prob"])
catalogue

,intervention,cost_usd_per_learner,tutor_minutes,assumed_base_uplift_pass_prob
0,automated_nudges,2.0,0,0.02
1,study_plan_coaching,15.0,20,0.06
2,tutor_outreach_call,35.0,45,0.10
3,peer_study_group,8.0,10,0.04


Costs and uplifts are **planning assumptions** (no causal data exists in OULAD); they are used only in the
prescriptive analysis and are varied in a sensitivity analysis in Module 8. The value of one additional pass
is assumed to be **USD 400** (contribution margin / retained fee).

## 2. Data acquisition and integrity
OULAD (Kuzilek et al., 2017, CC BY 4.0) — the Kaggle mirror required by the brief has identical content to the
UCI archive used here. File hashes are checked against the manifest recorded at download time.

In [3]:
manifest = json.loads((C.RAW / "raw_manifest.json").read_text())
rows = []
for name, meta in manifest["files"].items():
    p = C.RAW / name
    if p.suffix == ".csv" and p.exists():
        rows.append({"file": name, "bytes": p.stat().st_size, "sha256_matches_manifest": U.sha256(p) == meta["sha256"]})
integrity = pd.DataFrame(rows); integrity

,file,bytes,sha256_matches_manifest
0,assessments.csv,8211,True
1,courses.csv,526,True
2,studentAssessment.csv,5690483,True
3,studentInfo.csv,3462763,True
4,studentRegistration.csv,1132550,True
5,studentVle.csv,453836331,True
6,vle.csv,270612,True


In [4]:
t = oulad.read_raw()
shapes = pd.DataFrame([(k, *v.shape) for k, v in t.items()], columns=["table", "rows", "columns"])
shapes.loc[len(shapes)] = ["studentVle", 10_655_280, 6]
U.save_table(shapes, "m1_source_tables"); shapes

,table,rows,columns
0,courses,22,3
1,assessments,206,6
2,studentInfo,32593,12
3,studentRegistration,32593,5
4,studentAssessment,173912,5
5,vle,6364,6
6,studentVle,10655280,6


### Measurement scales (Lecture 1 / Lecture 7)
Knowing the scale decides the allowed operations and the encoding later on.

In [5]:
scales = pd.DataFrame([
    ("id_student, id_site, id_assessment", "Nominal (identifier)", "excluded from distances"),
    ("code_module, region, gender, disability, activity_type", "Nominal / categorical", "one-hot (profiling only)"),
    ("final_result", "Nominal outcome", "external validation only"),
    ("age_band, highest_education, imd_band", "Ordinal", "ordinal codes (profiling only)"),
    ("date, date_registration, deadlines (course-relative days)", "Interval", "differences only"),
    ("sum_click, score, session_minutes, counts", "Ratio", "log / power transform + z-score"),
], columns=["fields", "scale", "treatment"]); scales

,fields,scale,treatment
0,"id_student, id_site, id_assessment",Nominal (identifier),excluded from distances
1,"code_module, region, gender, disability, activ...",Nominal / categorical,one-hot (profiling only)
2,final_result,Nominal outcome,external validation only
3,"age_band, highest_education, imd_band",Ordinal,ordinal codes (profiling only)
4,"date, date_registration, deadlines (course-rel...",Interval,differences only
5,"sum_click, score, session_minutes, counts",Ratio,log / power transform + z-score


## 3. Big-data ETL with PySpark (Lectures 5–6)
`studentVle.csv` has 10.6M rows (454 MB). Spark reads it with an explicit schema, broadcasts the small `vle`
dimension (activity types), and aggregates with the **DataFrame API** to one row per enrollment × day. A
**Pair RDD** job (`map → reduceByKey`) recomputes total clicks per enrollment as an independent check.

In [6]:
t0 = time.time()
etl = etl_spark.run(prefer_spark=True)
print(f"ETL finished in {time.time()-t0:.0f}s"); etl

ETL finished in 121s


{'engine': 'pyspark',
 'studentVle_rows': 10655280,
 'studentVle_exact_duplicate_rows': 787170,
 'rows_without_site_metadata': 0,
 'enrollment_day_rows': 1808119,
 'enrollments_with_clicks': 29228,
 'total_clicks_dataframe': 39605099,
 'total_clicks_pair_rdd': 39605099,
 'pair_rdd_keys': 29228,
 'seconds': 117.3}

In [7]:
assert etl["total_clicks_dataframe"] == etl.get("total_clicks_pair_rdd", etl["total_clicks_dataframe"])
day = pd.read_parquet(C.INTERIM / "enrollment_day.parquet")
print(day.shape); day.head()

(1808119, 10)


,id_student,code_module,code_presentation,date,clicks,n_records,forum_clicks,content_clicks,quiz_clicks,distinct_sites
0,6516,AAA,2014J,-23,28,6,0,25,0,6
1,6516,AAA,2014J,-22,82,7,33,36,0,6
2,6516,AAA,2014J,-20,41,12,13,16,0,10
3,6516,AAA,2014J,-17,7,4,0,5,0,4
4,6516,AAA,2014J,-12,2,2,1,0,0,2


## 4. Integration at enrollment grain
Grain = (id_student, code_module, code_presentation): 32,593 enrollments of 28,785 learners. Activity and
assessments are aggregated **separately** and joined one-to-one, so joins never multiply rows
(`validate="one_to_one"` asserts this).

In [8]:
enr = oulad.add_derived(oulad.build_enrollment_table(t))
assert enr.duplicated(C.KEY).sum() == 0 and len(enr) == len(t["studentInfo"])
enr.to_parquet(C.INTERIM / "oulad_enrollments_raw.parquet", index=False)
print(enr.shape, "| learners:", enr.id_student.nunique())
enr[C.KEY + ["final_result", "total_clicks", "active_days", "avg_score_scheduled", "deadline_click_ratio"]].head()

(32593, 66) | learners: 28785


,id_student,code_module,code_presentation,final_result,total_clicks,active_days,avg_score_scheduled,deadline_click_ratio
0,11391,AAA,2013J,Pass,836.0,39.0,82.0,2.188038
1,28400,AAA,2013J,Pass,1220.0,73.0,66.4,0.308689
2,30268,AAA,2013J,Withdrawn,179.0,6.0,0.0,0.000000
3,31604,AAA,2013J,Pass,1989.0,118.0,76.0,1.695056
4,32885,AAA,2013J,Pass,739.0,62.0,54.4,0.873613


### Baseline KPIs (before any segmentation)

In [9]:
at_risk = enr["at_risk_outcome"].mean()
kpi_course = (enr.groupby(["course_domain", "code_module"])
              .agg(enrollments=("id_student", "size"), at_risk_rate=("at_risk_outcome", "mean"),
                   withdrawal_rate=("is_withdrawn", "mean"), no_activity_rate=("no_activity", "mean"))
              .round(3).reset_index())
blanket_cost = enr["at_risk_outcome"].sum() * C.INTERVENTION_CATALOGUE["tutor_outreach_call"][0]
print(f"At-risk learner rate: {at_risk:.1%}")
print(f"Blanket tutor-call cost if every eventually-at-risk learner were called: ${blanket_cost:,.0f} "
      f"(and at-risk status is only known at the end, so this is not actionable)")
U.save_metrics("m1", {"n_enrollments": len(enr), "n_learners": int(enr.id_student.nunique()), "at_risk_rate": at_risk,
                      "withdrawal_rate": enr.is_withdrawn.mean(), "pass_rate": enr.is_pass.mean(),
                      "no_activity_enrollments": int(enr.no_activity.sum()), "blanket_cost": blanket_cost, "etl": etl})
U.save_table(kpi_course, "m1_kpi_by_course"); kpi_course

At-risk learner rate: 52.8%
Blanket tutor-call cost if every eventually-at-risk learner were called: $602,280 (and at-risk status is only known at the end, so this is not actionable)


,course_domain,code_module,enrollments,at_risk_rate,withdrawal_rate,no_activity_rate
0,STEM,CCC,4434,0.622,0.445,0.133
1,STEM,DDD,6272,0.584,0.359,0.114
2,STEM,EEE,2934,0.438,0.246,0.106
3,STEM,FFF,7762,0.530,0.310,0.110
4,Social Sciences,AAA,748,0.290,0.168,0.027
5,Social Sciences,BBB,7909,0.525,0.302,0.179
6,Social Sciences,GGG,2534,0.403,0.115,0.078


## 5. Synthetic behavioural extension

OULAD lacks login sessions, time of day, video completion, forum **posts** and hour-level submission
timeliness. We simulate them with **conditional generation anchored to each learner's real record**:

| Required behaviour (brief) | Synthetic field(s) | Anchor in real OULAD data |
|---|---|---|
| Login frequency | sessions per active day (log-normal habit × clicks) + check-in logins | real active VLE days and daily clicks |
| Time-of-day pattern | `login_hour` via chronotype mixture | age band; deadline-driven clicking (cramming trait) |
| Video completion rate | `videos_started`, `videos_completed` | content clicks; engagement & score ranks |
| Days enrollment → first activity | first `login_ts` vs `date_registration` | real first active day |
| Forum / discussion participation | `forum_posts`, `forum_replies` | real `forumng` clicks × sociability |
| Assignment submission timeliness | `submit_hour`, `hours_before_deadline` | real submission and deadline days |

Faker supplies session UUIDs and user-agent strings; numpy (seed 42) supplies all distributions. Realistic
**defects are injected on purpose** (duplicates, label noise, missing and impossible values, 0.4% bot accounts)
so Module 3 can prove its cleaning rules against known ground truth.

In [10]:
t0 = time.time()
syn = synthetic.run(enr, day, t)
print(f"generated in {time.time()-t0:.0f}s"); syn

generated in 43s


{'sessions_clean_rows': 2580034,
 'sessions_with_defects': 2709570,
 'submission_events': 167044,
 'defects': {'bot_sessions': 108031,
  'messy_device_labels': 215045,
  'missing_session_minutes': 53761,
  'missing_device_type': 26880,
  'negative_minutes': 5376,
  'over_24h_minutes': 2688,
  'completed_gt_started': 8064,
  'exact_duplicate_rows': 21505,
  'final_rows': 2709570}}

In [11]:
sessions = pd.read_parquet(C.SYNTHETIC / "synthetic_login_sessions.parquet")
subs = pd.read_parquet(C.SYNTHETIC / "synthetic_submission_events.parquet")
traits = pd.read_parquet(C.SYNTHETIC / "_latent_traits_audit.parquet")
audit = pd.read_parquet(C.SYNTHETIC / "_session_source_audit.parquet")
print(sessions.shape, subs.shape); sessions.head()

(2709570, 14) (167044, 9)


,session_id,id_student,code_module,code_presentation,course_day,login_ts,login_hour,device_type,user_agent,session_minutes,videos_started,videos_completed,forum_posts,forum_replies
0,6af20cdc-72d8-42c9-961b-ea27965fd1a5,2689958,AAA,2013J,205,2014-04-24 13:30:02,13,desktop,Mozilla/5.0 (Linux; Android 2.3.1) AppleWebKit...,22.6,0,0,1,0
1,0cfb73e6-35a7-4abc-bb77-c1167801b605,2485582,DDD,2013J,195,2014-04-14 09:38:47,9,desktop,Mozilla/5.0 (iPod; U; CPU iPhone OS 3_3 like M...,41.8,1,1,0,0
2,5186c169-aeb1-43c6-896a-a66672950008,282317,DDD,2013J,149,2014-02-27 23:56:56,23,mobile,Mozilla/5.0 (compatible; MSIE 6.0; Windows 98;...,30.0,0,0,3,1
3,7047c347-165a-4af6-9e46-c1f6f8b20f6b,629540,FFF,2014B,147,2014-06-28 19:53:35,19,tablet,Mozilla/5.0 (iPod; U; CPU iPhone OS 4_2 like M...,219.7,0,0,0,0
4,dcad62c1-1614-4df5-8c17-dfd74baf6b12,684532,EEE,2014J,176,2015-03-26 05:49:51,5,tablet,Mozilla/5.0 (iPod; U; CPU iPhone OS 3_1 like M...,7.6,0,0,0,0


## 6. Validating the realism of the synthetic data
Checks performed on the log after excluding the injected bot accounts (the other injected defects are still
present, which is why some plausibility checks are just below 100% — Module 3 removes them):
1. **Range / plausibility** — hours 0–23, durations positive and right-skewed, completed ≤ started.
2. **Coherence with real behaviour** — every real active VLE day has ≥ 1 login; synthetic aggregates correlate
   with their real anchors *but not perfectly* (noise keeps them informative rather than copies).
3. **Behavioural face validity** — learners who eventually pass complete more videos and submit earlier.

In [12]:
clean = sessions.merge(audit, on="session_id")
clean = clean[clean["source"] != "bot"].drop_duplicates("session_id")
plaus = pd.DataFrame({
    "check": ["login_hour in 0..23", "session_minutes > 0 (non-missing)", "median session minutes",
              "videos_completed <= videos_started", "sessions per enrollment (median)"],
    "value": [clean.login_hour.between(0, 23).mean(), (clean.session_minutes.dropna() > 0).mean(),
              clean.session_minutes.median(), (clean.videos_completed <= clean.videos_started).mean(),
              clean.groupby(C.KEY).size().median()]})
plaus

,check,value
0,login_hour in 0..23,1.000000
1,session_minutes > 0 (non-missing),0.997952
2,median session minutes,15.200000
3,videos_completed <= videos_started,0.996874
4,sessions per enrollment (median),57.000000


In [13]:
agg = clean.groupby(C.KEY).agg(logins=("session_id", "size"), videos_started=("videos_started", "sum"),
                                videos_completed=("videos_completed", "sum"), posts=("forum_posts", "sum"),
                                night=("login_hour", lambda h: h.isin([22, 23, 0, 1, 2, 3, 4, 5]).mean())).reset_index()
chk = enr.merge(agg, on=C.KEY, how="left").fillna({"logins": 0, "posts": 0})
chk["vcr"] = chk.videos_completed / chk.videos_started.replace(0, np.nan)
sub_agg = subs.groupby(C.KEY)["hours_before_deadline"].median().rename("median_hbd").reset_index()
chk = chk.merge(sub_agg, on=C.KEY, how="left")
anchors = pd.DataFrame([
    ("logins vs real active days", chk[["logins", "active_days"]].corr("spearman").iloc[0, 1]),
    ("forum posts vs real forum clicks", chk[["posts", "forum_clicks"]].corr("spearman").iloc[0, 1]),
    ("video completion vs real avg score", chk[["vcr", "avg_score_scheduled"]].corr("spearman").iloc[0, 1]),
    ("median hours before deadline vs real mean days early", chk[["median_hbd", "mean_days_early"]].corr("spearman").iloc[0, 1]),
    ("night-login share vs real deadline click ratio", chk[["night", "deadline_click_ratio"]].corr("spearman").iloc[0, 1]),
], columns=["synthetic vs real anchor", "spearman_rho"]).round(3)
U.save_table(anchors, "m1_synthetic_anchor_correlations"); anchors

,synthetic vs real anchor,spearman_rho
0,logins vs real active days,0.986
1,forum posts vs real forum clicks,0.819
2,video completion vs real avg score,0.489
3,median hours before deadline vs real mean days...,0.911
4,night-login share vs real deadline click ratio,0.215


In [14]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
tr = traits.set_index(C.KEY)
h = clean.join(tr["trait_chronotype"], on=C.KEY)
for ch, col in zip(["morning", "afternoon", "evening", "night"], V.CATEGORICAL):
    ax[0].hist(h.loc[h.trait_chronotype == ch, "login_hour"], bins=24, range=(0, 24), histtype="step", lw=2, color=col, label=ch, density=True)
ax[0].set(title="Login hour by simulated chronotype", xlabel="hour of day", ylabel="density"); ax[0].legend()
ax[1].hist(np.log10(clean.session_minutes.dropna().clip(lower=0.1)), bins=60, color=V.CATEGORICAL[0])
ax[1].set(title="Session duration (log10 minutes)", xlabel="log10(minutes)", ylabel="sessions")
order = ["Distinction", "Pass", "Fail", "Withdrawn"]
data = [chk.loc[chk.final_result == o, "vcr"].dropna() for o in order]
b = ax[2].boxplot(data, tick_labels=order, showfliers=False, patch_artist=True)
for patch, o in zip(b["boxes"], order): patch.set_facecolor(V.OUTCOME_COLORS[o]); patch.set_alpha(0.6)
ax[2].set(title="Video completion rate by real final result", ylabel="completed / started")
V.save(fig, "m1_synthetic_validation"); plt.show()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4416\669669051.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  V.save(fig, "m1_synthetic_validation"); plt.show()


In [15]:
fig, ax = plt.subplots(figsize=(7, 4))
x = subs.merge(traits[C.KEY + ["trait_cramming"]], on=C.KEY)
x["cram_group"] = pd.cut(x.trait_cramming, [0, 0.33, 0.66, 1.0], labels=["low", "medium", "high"], include_lowest=True)
for g, col in zip(["low", "medium", "high"], [V.CATEGORICAL[0], V.CATEGORICAL[3], V.CATEGORICAL[1]]):
    v = x.loc[(x.cram_group == g) & x.hours_before_deadline.between(-48, 168), "hours_before_deadline"]
    ax.hist(v, bins=72, histtype="step", lw=2, density=True, label=f"{g} cramming propensity", color=col)
ax.axvline(0, color=V.TEXT2, lw=1, ls="--"); ax.axvline(24, color=V.TEXT2, lw=1, ls=":")
ax.set(title="Submission lead time (hours before 23:59 deadline)", xlabel="hours before deadline (negative = late)", ylabel="density")
ax.legend(); V.save(fig, "m1_submission_lead_time"); plt.show()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4416\1225152668.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(); V.save(fig, "m1_submission_lead_time"); plt.show()


**Reading the checks.**
* **Logins vs real active days (ρ ≈ 0.99)** — by design: a login is required for every active day. Login frequency
  is therefore consistent with the real data but redundant with `active_day_fraction`; it is kept for profiling and
  the API, not as a separate clustering input.
* **Hours before deadline vs real days early (ρ ≈ 0.91)** — the day is real, the synthetic part is the *hour*,
  which is what separates a submission at 09:00 from one at 23:30 on deadline day (the "last-minute" signal).
* **Forum posts vs forum clicks (ρ ≈ 0.82)** — posting is proportional to forum presence but varies by sociability.
* **Video completion vs score (ρ ≈ 0.49)** and **night logins vs deadline clicking (ρ ≈ 0.22)** — moderate links:
  these fields add behavioural information rather than copying OULAD.

Deadline-driven learners submit inside the last 24 hours far more often, the behaviour the brief asks us to capture.

**Limitations.** Relationships between synthetic fields and outcomes are partly created by the generation logic
(e.g., video propensity depends on score rank). They are therefore *plausible*, not *evidence*; Module 5 checks that
the segmentation does not rely on synthetic fields alone (an OULAD-only model is compared).

In [16]:
U.save_metrics("m1", {"synthetic": syn, "anchor_correlations": dict(zip(anchors.iloc[:, 0], anchors.iloc[:, 1])),
                      "plausibility": dict(zip(plaus.check, plaus.value))})

## 7. Data dictionary
Every synthetic field is documented with name, type, unit, valid range and generation logic / assumption
(`docs/DATA_DICTIONARY.xlsx` and `docs/DATA_DICTIONARY.md`).

In [17]:
dd = data_dictionary.build()
print(len(dd), "documented fields"); dd.head(12)

51 documented fields


,table,column,source,dtype,unit,valid_range,generation_logic_or_definition,business_assumption,role
0,synthetic_login_sessions,session_id,Synthetic,string (UUID4),-,unique,"Faker.uuid4() (seeded, Faker.seed(42)).",One row per authenticated session.,primary key
1,synthetic_login_sessions,id_student,OULAD key,int,-,existing studentInfo id,Copied from OULAD enrollment.,,foreign key
2,synthetic_login_sessions,code_module,OULAD key,string,-,AAA..GGG,Copied from OULAD enrollment.,,foreign key
3,synthetic_login_sessions,code_presentation,OULAD key,string,-,2013B..2014J,Copied from OULAD enrollment.,,foreign key
4,synthetic_login_sessions,course_day,Synthetic (anchored),int,days relative to course start,-30 .. L,Days with real VLE clicks receive 1 + Poisson(...,Every day with recorded clicks requires at lea...,raw
5,synthetic_login_sessions,login_ts,Synthetic,datetime,local time,within presentation calendar,"Presentation start (1 Feb for B / 1 Oct for J,...",Exact OU start dates are not published; 1st of...,raw
6,synthetic_login_sessions,login_hour,Synthetic,int,hour of day,0..23,floor((chronotype centre + 2.5 x cramming x ne...,Younger learners study later; deadline pressur...,raw
7,synthetic_login_sessions,device_type,Synthetic,category,-,desktop | mobile | tablet (clean),75% learner's primary device (Categorical by a...,Device mix by age reflects common LMS usage pa...,raw
8,synthetic_login_sessions,user_agent,Synthetic,string,-,-,"Sampled from a pool of 3,000 Faker.user_agent(...",,raw
9,synthetic_login_sessions,session_minutes,Synthetic,float,minutes,0 < x <= 1440 (clean),"exp(N(log(6 + 1.1 x clicks_in_session), 0.45))...",Duration grows with interaction volume; log-no...,raw
